In [11]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq
from datasets import Dataset
from tqdm import tqdm 

In [22]:
model_path = "G:\\model_weights\\models\\Qwen\\qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda:0")

In [4]:
def generate(inputs):
    s = time.time()
    with torch.no_grad():
        outputs = model(**inputs)
    e = time.time()

    logits = outputs.logits
    last_logits = logits[:, -1, :]
    next_token_id = last_logits.argmax(dim=-1)
    # return next_token_id, outputs.past_key_values, e - s
    return e - s

In [5]:
prompts = [
    "春风又绿江南岸",
    "FROM fairest creatures we desire increase,",
    "咖啡馆角落里的老人静静读着报纸，时光仿佛在他身边慢了下来。海浪一遍遍拍打着礁石，像是在诉说一个古老而永恒的故事。",
    "小时候总以为长大就能解决所有问题，如今才知并非如此。夜深人静时，我常常望着窗外的月亮，思绪飘向远方。那只流浪猫每天准时出现在门口，仿佛在等待某个人归来。春天来了，樱花纷纷扬扬地落下，铺满了整条街道，如梦似幻。",
    "评价一下这首诗《将进酒》君不见黄河之水天上来，奔流到海不复回。君不见高堂明镜悲白发，朝如青丝暮成雪。人生得意须尽欢，莫使金樽空对月。天生我材必有用，千金散尽还复来。烹羊宰牛且为乐，会须一饮三百杯。岑夫子，丹丘生，将进酒，杯莫停。与君歌一曲，请君为我倾耳听。钟鼓馔玉不足贵，但愿长醉不愿醒。古来圣贤皆寂寞，惟有饮者留其名。陈王昔时宴平乐，斗酒十千恣欢谑。主人何为言少钱，径须沽取对君酌。五花马，千金裘，呼儿将出换美酒，与尔同销万古愁。"
]

In [21]:
prompt_length = []
latency_collect = []

In [25]:
print("Token lengths of the prompts:")
for prompt in prompts:
    print(tokenizer(prompt).input_ids.__len__())
    inputs = tokenizer(prompt, return_tensors="pt")
    prompt_length.append(inputs.input_ids.shape[1])
    latency_collect.append(generate(inputs))

Token lengths of the prompts:
5
8
36
69
199


In [27]:
latency_collect, torch.tensor(latency_collect) / torch.tensor(prompt_length) 

([0.2712733745574951,
  0.3133969306945801,
  0.5044102668762207,
  0.6170060634613037,
  0.9914960861206055],
 tensor([0.0543, 0.0392, 0.0140, 0.0089, 0.0050]))

In [19]:
text_data_path = "./sonnet.txt"
with open(text_data_path, "r", encoding="utf-8") as f:
    text_data = f.readlines()


In [38]:
chat = [
    {"role": "system", "content": "你是我的人工智能助手，协助我完成各种任务。"},
    {"role": "user", "content": "prompt"}
]

In [ ]:
template = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
template = '<|im_start|>system\n你是我的人工智能助手，协助我完成各种任务。<|im_end|>\n<|im_start|>user\nprompt<|im_end|>\n<|im_start|>assistant\n'

'<|im_start|>system\n你是我的人工智能助手，协助我完成各种任务。<|im_end|>\n<|im_start|>user\nprompt<|im_end|>\n<|im_start|>assistant\n'

In [95]:
text_data[:5]

['FROM fairest creatures we desire increase,\n',
 "That thereby beauty's rose might never die,\n",
 'But as the riper should by time decease,\n',
 'His tender heir might bear his memory:\n',
 'But thou, contracted to thine own bright eyes,\n']

In [20]:
test_data = []
for i in text_data:
    # chat = [
    #     {"role": "system", "content": "你是我的人工智能助手，协助我完成各种任务。"},
    #     {"role": "user", "content": i} 
    # ]
    # prompt = template.replace("prompt", i.strip())
    # prompt = tokenizer.apply_chat_template(i, add_generation_prompt=True, tokenize=False, padding_side="left")
    ...
test_data = tokenizer(text_data[:512],max_length=40, truncation=True, padding_side="left", padding=True, return_tensors="pt").to("cuda:0")

In [8]:
print(test_data['input_ids'].shape)

torch.Size([512, 16])


In [ ]:
# ds = Dataset.from_list(test_data[:512])

In [84]:
for i in ds[:5]['input_ids']:
    print(len(i))

32
33
35
32
35


In [ ]:
from collections import defaultdict
throughtput_collect = {}

In [ ]:
step = [4, 8, 16, 32]
for s in step:
    throughtput_collect[s] = {"total_time": 0, "total_tokens": 0, "prompt_tokens": 0, "generate_tokens":0}
    start = time.time()
    for i in tqdm(range(0, 512, s)):
        slice_ = slice(i, i + s)
        inputs = {
            "input_ids": test_data['input_ids'][slice_],
            "attention_mask": test_data['attention_mask'][slice_]
        }
        outputs = model.generate(**inputs)
        prompt_tokens = inputs['input_ids'].shape[1] * inputs['input_ids'].shape[0]
        all_tokens = outputs.shape[1] * outputs.shape[0]
        generate_tokens = all_tokens - prompt_tokens
        throughtput_collect[s]["total_tokens"] += all_tokens
        throughtput_collect[s]["prompt_tokens"] += prompt_tokens
        throughtput_collect[s]["generate_tokens"] += generate_tokens
    throughtput_collect[s]["total_time"] += time.time() - start

100%|██████████| 16/16 [00:11<00:00,  1.38it/s]


In [23]:
step = [4, 8, 16, 32]
step = [1]
for s in step:
    for i in range(0, 512, s):
        slice_ = slice(i, i + s)
        inputs = {
            "input_ids": test_data['input_ids'][slice_],
            "attention_mask": test_data['attention_mask'][slice_]
        }
        start = time.time()
        outputs = model.generate(**inputs)
        print("latency:",time.time() - start)

latency: 0.8161122798919678
latency: 0.5158741474151611
latency: 0.5256617069244385
latency: 0.5332634449005127
latency: 0.5274229049682617
latency: 0.5312914848327637
latency: 0.5245330333709717


KeyboardInterrupt: 

In [15]:
throughtput_collect

{32: {'total_time': 11.617451906204224,
  'total_tokens': 576,
  'prompt_tokens': 256,
  'generate_tokens': 320},
 4: {'total_time': 84.72974228858948,
  'total_tokens': 4608,
  'prompt_tokens': 2048,
  'generate_tokens': 2560},
 8: {'total_time': 42.75325298309326,
  'total_tokens': 2304,
  'prompt_tokens': 1024,
  'generate_tokens': 1280},
 16: {'total_time': 21.498729705810547,
  'total_tokens': 1152,
  'prompt_tokens': 512,
  'generate_tokens': 640}}

In [24]:
throughtput_collect

{32: {'total_time': 11.617451906204224,
  'total_tokens': 576,
  'prompt_tokens': 256,
  'generate_tokens': 320},
 4: {'total_time': 84.72974228858948,
  'total_tokens': 4608,
  'prompt_tokens': 2048,
  'generate_tokens': 2560},
 8: {'total_time': 42.75325298309326,
  'total_tokens': 2304,
  'prompt_tokens': 1024,
  'generate_tokens': 1280},
 16: {'total_time': 21.498729705810547,
  'total_tokens': 1152,
  'prompt_tokens': 512,
  'generate_tokens': 640}}